# 데이터셋 코드 테스트 — `pm_safeline`

PM 세이프라인 데이터셋 파이프라인(사고 → OSM 지점 → negative → 로드뷰 이미지 → split)을
단계별로 검증하고 **데이터 개수**를 집계한다.

- 사고 데이터셋: `datasets.accidents` (KoROAD/TAAS)
- 데이터 primitive: `datasets.primitives` (geo/negatives/streetview)
- 로드뷰 데이터셋: `datasets.roadview`

> 수집 단계는 **torch 불필요**. 이미지 수집은 `mock` provider 로 검증.

In [1]:
import os, sys, dataclasses
sys.path.insert(0, os.path.abspath(".."))
os.environ["PM_SV_PROVIDER"] = "mock"
import pandas as pd
from pm_safeline.datasets.primitives.config import Config
cfg = Config()
print("data_dir =", cfg.data_dir, "| 대전 유효범위 =", cfg.bbox)

data_dir = C:\Users\BREW\Downloads\AI기반사회문제해결프로젝트\proj\data | 대전 유효범위 = (127.24, 36.17, 127.57, 36.51)


## 1. 사고 데이터셋 — `load_accidents`

KoROAD 이륜차 다발지역을 표준 스키마로 로드. **사고 좌표는 자르지 않고 전부 사용**하되,
대전 밖 좌표(데이터 오류)만 제외한다.

In [2]:
from pm_safeline.datasets.accidents import load_accidents, CORE_COLUMNS
acc = load_accidents("koroad", cfg, download=False)
n_pos = len(acc)
print(f"사고 지점(positive): {n_pos}")
print("severity 분포:", acc["severity"].value_counts().to_dict())
acc[["accident_id","lat","lon","severity","mode"]].head()

[accidents] 대전 범위 밖 좌표(오류 의심) 1건 제외
사고 지점(positive): 49
severity 분포: {'중상': 37, '경상': 9, '사망': 3}


,accident_id,lat,lon,severity,mode
0,2,36.345429,127.445795,경상,motorcycle_frequentzone
1,3,36.347315,127.431974,경상,motorcycle_frequentzone
2,4,36.353050,127.368831,경상,motorcycle_frequentzone
3,5,36.351410,127.378420,경상,motorcycle_frequentzone
4,6,36.352113,127.376193,경상,motorcycle_frequentzone


## 2. 작업 영역 = 사고 좌표들의 실제 범위 (+여유)

In [3]:
minx,miny,maxx,maxy = acc.total_bounds
region = (minx-0.01, miny-0.01, maxx+0.01, maxy+0.01)
cfg = dataclasses.replace(cfg, bbox=region)   # OSM/negative 는 이 영역에서
print("작업 영역(W,S,E,N):", tuple(round(v,3) for v in region))

작업 영역(W,S,E,N): (np.float64(127.328), np.float64(36.302), np.float64(127.466), np.float64(36.375))


## 3. `geo` — OSM 도로망 → 지점/스냅 (작업 영역 기준)

In [4]:
from pm_safeline.datasets.primitives import geo
edges = geo.load_drive_edges(cfg)
candidates = geo.sample_points_along_edges(edges, cfg)
acc_snapped = geo.snap_accidents_to_edges(acc, edges, cfg)
n_edges, n_cand = len(edges), len(candidates)
print(f"OSM edges: {n_edges} · 도로 후보 지점: {n_cand}")

OSM edges: 27972 · 도로 후보 지점: 80876


## 4. `negatives` — exposure-matched 대조지점

In [5]:
from pm_safeline.datasets.primitives import negatives
negs = negatives.sample_negatives(acc_snapped, candidates, cfg)
labeled = negatives.build_labeled_points(acc_snapped, negs)
n_neg = int((labeled.label==0).sum())
print(f"대조 지점(negative): {n_neg}  (ratio={cfg.negative_ratio})")
print(f"총 라벨 지점: {len(labeled)}  (pos {int((labeled.label==1).sum())} / neg {n_neg})")
labeled[["point_id","label","lat","lon","heading"]].head()

대조 지점(negative): 147  (ratio=3.0)
총 라벨 지점: 196  (pos 49 / neg 147)


,point_id,label,lat,lon,heading
0,acc_000000,1,36.345429,127.445795,60.7
1,acc_000001,1,36.347315,127.431974,61.1
2,acc_000002,1,36.353050,127.368831,129.5
3,acc_000003,1,36.351410,127.378420,294.4
4,acc_000004,1,36.352113,127.376193,114.9


## 5. 로드뷰 이미지 수집 (mock, 소량)

In [6]:
import tempfile
from pathlib import Path
from pm_safeline.datasets import roadview
# 데모(mock) 이미지는 임시 폴더에만 — 실제 data/ 를 오염시키지 않음
cfg_demo = dataclasses.replace(cfg, data_dir=Path(tempfile.mkdtemp()) / 'data')
import os; os.environ['PM_SV_PROVIDER']='mock'
subset = labeled.head(8).copy()
manifest = roadview.collect_images(subset, cfg_demo)
print(f'데모 수집(mock, 임시폴더): {len(manifest)} 이미지 / {subset.point_id.nunique()} 지점')
# 지점당 4방향 수집 → 전체 예상 이미지 수
n_img_full = len(labeled) * 4
print(f'전체 수집 시 총 이미지(지점 {len(labeled)} x 4방향): {n_img_full}')

[collect] 8/8 지점 처리
[collect] manifest 저장: C:\Users\Public\Documents\ESTsoft\CreatorTemp\tmpbze9i1l5\data\manifest.csv (32 이미지)
데모 수집(mock, 임시폴더): 32 이미지 / 8 지점
전체 수집 시 총 이미지(지점 196 x 4방향): 784


## 6. 학습 분할 — 지점 누수 방지 + 라벨 stratify (torch 불필요)

In [7]:
from pm_safeline.datasets.roadview import split_indices, kfold_indices
tr, va = split_indices(labeled, valid_frac=0.2, seed=42)
tp=set(labeled.iloc[tr].point_id); vp=set(labeled.iloc[va].point_id)
print(f"train {len(tr)}지점 / valid {len(va)}지점 · 지점누수={tp&vp}")
folds=kfold_indices(labeled,n_splits=5,seed=1)
leak=any(set(labeled.iloc[a].point_id)&set(labeled.iloc[b].point_id) for a,b in folds)
print("k-fold(5) 누수:", leak, "→ False 정상")

train 156지점 / valid 40지점 · 지점누수=set()


k-fold(5) 누수: False → False 정상


## 7. 요약 — 데이터 개수

In [8]:
summary = pd.DataFrame([
    ("사고 지점 (positive)", n_pos),
    ("대조 지점 (negative)", n_neg),
    ("총 라벨 지점", len(labeled)),
    ("총 로드뷰 이미지 (수집 시)", n_img_full),
    ("OSM edges", n_edges),
    ("도로 후보 지점", n_cand),
], columns=["항목","개수"])
print(summary.to_string(index=False))

              항목    개수
사고 지점 (positive)    49
대조 지점 (negative)   147
         총 라벨 지점   196
총 로드뷰 이미지 (수집 시)   784
       OSM edges 27972
        도로 후보 지점 80876
